# Model Training

Benchmarks 7 regression models (Linear Regression, Random Forest, Extra Trees, Gradient Boosting, XGBoost, LightGBM, MLP) to predict NDMI (soil moisture proxy) from 10 satellite-derived features, using a time-based split (train: 2018–2021, test: 2022).

In [ ]:
# Cell 6: PREPARE TRAINING ARRAYS

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# Load FINAL dataset
BASE_PATH = "/content/drive/MyDrive/mini_project_1"
csv_path = f"{BASE_PATH}/Final_dataset.csv"

df = pd.read_csv(csv_path)
print("Dataset shape:", df.shape)
print("Year range:", df['year'].min(), "-", df['year'].max())

# Drop rows where target is NaN
df = df.dropna(subset=['ndmi'])

#feature names
features = [
    'ndvi',
    'lst',
    'precip',
    'smap',
    'vv',
    'vh',
    'vv_vh_ratio',
    'ch4',
    'year',
    'month'
]

target = 'ndmi'

# TIME-BASED SPLIT
# Train: 2018–2021
# Test: 2022

train_df = df[df['year'] <= 2021]
test_df  = df[df['year'] == 2022]

X_train = train_df[features].values
y_train = train_df[target].values

X_test = test_df[features].values
y_test = test_df[target].values

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# SCALING (FIT ONLY ON TRAIN)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, f"{BASE_PATH}/scaler.joblib")

print("Preprocessing complete.")

Dataset shape: (123417, 11)
Year range: 2018 - 2022
Train shape: (97879, 10)
Test shape: (25538, 10)
Preprocessing complete.


## LightGBM (Best-Performing Model)

In [ ]:
# CELL 7A : Train LightGBM regression model

from lightgbm import LGBMRegressor
import joblib

# Define LightGBM model
lgbm = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)

# Train model
lgbm.fit(X_train, y_train)

# Save trained model
joblib.dump(lgbm, f"{BASE_PATH}/lightgbm_final.joblib")

print("LightGBM model saved to Drive.")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003932 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2056
[LightGBM] [Info] Number of data points in the train set: 97879, number of used features: 10
[LightGBM] [Info] Start training from score 0.001100
LightGBM model saved to Drive.


In [ ]:
# CELL 8A : Evaluate trained LightGBM model on both training and test datasets

import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib

# Load trained model
model = joblib.load(f"{BASE_PATH}/lightgbm_final.joblib")

# Train predictions
y_train_pred = model.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)

# Test predictions
y_test_pred = model.predict(X_test)
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

# Print results
print("________ TRAIN PERFORMANCE ________")
print("R2:", train_r2)
print("RMSE:", train_rmse)
print("MAE:", train_mae)

print("\n________ TEST PERFORMANCE ________")
print("R2:", test_r2)
print("RMSE:", test_rmse)
print("MAE:", test_mae)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


________ TRAIN PERFORMANCE ________
R2: 0.9736297031339662
RMSE: 0.021707913945019308
MAE: 0.015884142708120722

________ TEST PERFORMANCE ________
R2: 0.8922995478840459
RMSE: 0.041476739694151145
MAE: 0.02987157460776288


In [ ]:
# CELL 9A : Compute permutation importance on test set and derive DI weights

import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance
import joblib

BASE_PATH = "/content/drive/MyDrive/mini_project_1"   # corrected folder

# Load trained model
model = joblib.load(f"{BASE_PATH}/lightgbm_final.joblib")

# Feature names (must match training order)
feature_names = [
    'ndvi',
    'lst',
    'precip',
    'smap',
    'vv',
    'vh',
    'vv_vh_ratio',
    'ch4',
    'year',
    'month'
]

# Compute permutation importance on TEST SET
result = permutation_importance(
    model,
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='r2',
    n_jobs=-1
)

# Build importance DataFrame
imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
})

imp_df = imp_df.sort_values(by='importance_mean', ascending=False)

print("Permutation Importance (based on R2 drop):")
print(imp_df)

# Normalize positive importances for DI weights
imp_df['importance_positive'] = imp_df['importance_mean'].clip(lower=0)
total = imp_df['importance_positive'].sum()
imp_df['norm_weight'] = imp_df['importance_positive'] / total

print("\nNormalized Weights (for DI):")
print(imp_df[['feature','norm_weight']])

# Save importance file
imp_df.to_csv(f"{BASE_PATH}/feature_importance_lightgbm.csv", index=False)

print("\nSaved feature importance to Drive.")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Permutation Importance (based on R2 drop):
       feature  importance_mean  importance_std
0         ndvi         2.411030        0.011983
6  vv_vh_ratio         0.062616        0.001521
1          lst         0.000000        0.000000
3         smap         0.000000        0.000000
5           vh         0.000000        0.000000
4           vv         0.000000        0.000000
8         year         0.000000        0.000000
7          ch4         0.000000        0.000000
9        month         0.000000        0.000000
2       precip        -0.001056        0.000113

Normalized Weights (for DI):
       feature  norm_weight
0         ndvi     0.974687
6  vv_vh_ratio     0.025313
1          lst     0.000000
3         smap     0.000000
5           vh     0.000000
4           vv     0.000000
8         year     0.000000
7          ch4     0.000000
9        month     0.000000
2       precip     0.000000

Saved feature importance to Drive.


## MLP (Neural Network Baseline)

In [ ]:
# CELL 7B : Train MLP regression model

from sklearn.neural_network import MLPRegressor
import joblib

# Number of features
n_features = X_train_scaled.shape[1]

# Define MLP model
mlp = MLPRegressor(
            hidden_layer_sizes=(256,128,64),
            activation='relu',
            max_iter=1000,
            early_stopping=True,
            random_state=42
        )

mlp.fit(X_train_scaled, y_train)

# Save trained model
joblib.dump(mlp, f"{BASE_PATH}/dlmlp_final.joblib")

print("Model saved to Drive.")

Model saved to Drive.


In [ ]:
# CELL 8B : Evaluate trained MLP model on both training and test datasets

import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib

# Load trained model
model = joblib.load(f"{BASE_PATH}/dlmlp_final.joblib")

# Train predictions
y_train_pred = model.predict(X_train_scaled)
train_r2 = r2_score(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)

# Test predictions
y_test_pred = model.predict(X_test_scaled)
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

# Print results
print("________ TRAIN PERFORMANCE ________")
print("R2:", train_r2)
print("RMSE:", train_rmse)
print("MAE:", train_mae)

print("\n________ TEST PERFORMANCE ________")
print("R2:", test_r2)
print("RMSE:", test_rmse)
print("MAE:", test_mae)

________ TRAIN PERFORMANCE ________
R2: 0.9731739853873147
RMSE: 0.021894682926007526
MAE: 0.015931443225000768

________ TEST PERFORMANCE ________
R2: 0.8167553712706432
RMSE: 0.05410177010003507
MAE: 0.039888184446491046


In [ ]:
# CELL 9B : Compute permutation importance on test set and derive DI weights

import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance
import joblib

BASE_PATH = "/content/drive/MyDrive/mini_project_1"   # corrected folder

# Load trained model
model = joblib.load(f"{BASE_PATH}/dlmlp_final.joblib")

# Feature names (must match training order)
feature_names = [
    'ndvi',
    'lst',
    'precip',
    'smap',
    'vv',
    'vh',
    'vv_vh_ratio',
    'ch4',
    'year',
    'month'
]

# Compute permutation importance on TEST SET
result = permutation_importance(
    model,
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='r2',
    n_jobs=-1
)

# Build importance DataFrame
imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
})

imp_df = imp_df.sort_values(by='importance_mean', ascending=False)

print("Permutation Importance (based on R2 drop):")
print(imp_df)

# Normalize positive importances for DI weights
imp_df['importance_positive'] = imp_df['importance_mean'].clip(lower=0)
total = imp_df['importance_positive'].sum()
imp_df['norm_weight'] = imp_df['importance_positive'] / total

print("\nNormalized Weights (for DI):")
print(imp_df[['feature','norm_weight']])

# Save importance file
imp_df.to_csv(f"{BASE_PATH}/feature_importance_dlmlp .csv", index=False)

print("\nSaved feature importance to Drive.")

Permutation Importance (based on R2 drop):
       feature  importance_mean  importance_std
0         ndvi         1.331485        0.006977
1          lst         0.268086        0.002254
6  vv_vh_ratio         0.088332        0.001497
4           vv         0.043846        0.000544
5           vh         0.039085        0.001197
7          ch4         0.019884        0.000759
3         smap         0.016277        0.000597
8         year         0.000000        0.000000
9        month        -0.013450        0.000970
2       precip        -0.023599        0.001165

Normalized Weights (for DI):
       feature  norm_weight
0         ndvi     0.736850
1          lst     0.148360
6  vv_vh_ratio     0.048884
4           vv     0.024265
5           vh     0.021630
7          ch4     0.011004
3         smap     0.009008
8         year     0.000000
9        month     0.000000
2       precip     0.000000

Saved feature importance to Drive.


## Full Model Comparison

In [ ]:
# MODEL COMPARISON + SAVE TRAINED MODELS

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import numpy as np
import pandas as pd
import joblib

models = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),

    "XGBoost":
        xgb.XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ),

    "LightGBM":
        lgb.LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            random_state=42
        ),

    "MLP":
        MLPRegressor(
            hidden_layer_sizes=(256,128,64),
            activation='relu',
            max_iter=1000,
            early_stopping=True,
            random_state=42
        )
}

results = []

for name, model in models.items():

    print(f"Training {name}...")

    # Tree models do not need scaling
    if name in ["Random Forest","Extra Trees","Gradient Boosting","XGBoost","LightGBM"]:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

    else:
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)

    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)

    results.append([name, r2, rmse, mae])

    # Save trained model
    model_filename = name.lower().replace(" ", "_") + "_model.joblib"
    joblib.dump(model, f"{BASE_PATH}/{model_filename}")

    print(f"{name} model saved to Drive.")

results_df = pd.DataFrame(results, columns=["Model","R2","RMSE","MAE"])
print("\nModel Comparison:")
print(results_df.sort_values("R2", ascending=False))

Training Linear Regression...
Linear Regression model saved to Drive.
Training Random Forest...
Random Forest model saved to Drive.
Training Extra Trees...
Extra Trees model saved to Drive.
Training Gradient Boosting...
Gradient Boosting model saved to Drive.
Training XGBoost...
XGBoost model saved to Drive.
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003947 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2056
[LightGBM] [Info] Number of data points in the train set: 97879, number of used features: 10
[LightGBM] [Info] Start training from score 0.001100


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM model saved to Drive.
Training MLP...
MLP model saved to Drive.

Model Comparison:
               Model        R2      RMSE       MAE
5           LightGBM  0.892300  0.041477  0.029872
1      Random Forest  0.891158  0.041696  0.030451
4            XGBoost  0.885914  0.042689  0.031668
2        Extra Trees  0.880579  0.043675  0.031226
3  Gradient Boosting  0.872691  0.045095  0.032381
0  Linear Regression  0.825270  0.052830  0.038444
6                MLP  0.816755  0.054102  0.039888


In [ ]:
# Evaluate trained models on both training and test datasets

import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib

BASE_PATH = "/content/drive/MyDrive/mini_project_1"

# Load trained model
model = joblib.load(f"{BASE_PATH}/random_forest_model.joblib")

# Train predictions
y_train_pred = model.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)

# Test predictions
y_test_pred = model.predict(X_test)
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

# Print results
print("________ TRAIN PERFORMANCE ________")
print("R2:", train_r2)
print("RMSE:", train_rmse)
print("MAE:", train_mae)

print("\n________ TEST PERFORMANCE ________")
print("R2:", test_r2)
print("RMSE:", test_rmse)
print("MAE:", test_mae)

________ TRAIN PERFORMANCE ________
R2: 0.996178506533
RMSE: 0.00826374947436178
MAE: 0.005717997548591547

________ TEST PERFORMANCE ________
R2: 0.891157824999508
RMSE: 0.04169600572978612
MAE: 0.03045090393646159


In [ ]:
# CELL  : Compute permutation importance on test set and derive DI weights

import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance
import joblib

BASE_PATH = "/content/drive/MyDrive/mini_project_1"   # corrected folder

# Load trained model
model = joblib.load(f"{BASE_PATH}/random_forest_model.joblib")

# Feature names (must match training order)
feature_names = [
    'ndvi',
    'lst',
    'precip',
    'smap',
    'vv',
    'vh',
    'vv_vh_ratio',
    'ch4',
    'year',
    'month'
]

# Compute permutation importance on TEST SET
result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='r2',
    n_jobs=-1
)

# Build importance DataFrame
imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
})

imp_df = imp_df.sort_values(by='importance_mean', ascending=False)

print("Permutation Importance (based on R2 drop):")
print(imp_df)

# Normalize positive importances for DI weights
imp_df['importance_positive'] = imp_df['importance_mean'].clip(lower=0)
total = imp_df['importance_positive'].sum()
imp_df['norm_weight'] = imp_df['importance_positive'] / total

print("\nNormalized Weights (for DI):")
print(imp_df[['feature','norm_weight']])

# Save importance file
imp_df.to_csv(f"{BASE_PATH}/feature_importance_random_forest.csv", index=False)

print("\nSaved feature importance to Drive.")